# 06 - Per-degradation Robustness and Out-of-Distribution Evaluation
**Addresses comments 1 and 2.**

**Robustness (replaces Table VI).** The submitted test used `list_npy(TRAIN_GT_DIR)[:30]` - the first 30 files of the *training* folder, ~90% of which were training images - one noise seed per image, and hard-coded degradation strengths. Here: **test-split images only**, degradation strengths **estimated from the training pairs** (notebook 00), 3 independent noise draws per image, all seeds of every model, paired statistics.

**OOD (replaces Table VII / Fig. 10).** The submitted run read the first ten files of `data.zip`, which were `data/masks/*.png` - *segmentation masks, not SEM images*. The loader below separates images from masks, **refuses label-map-like files**, and writes a **manifest** (file names, SHA-256 prefixes, original shapes, selection seed) that must be quoted in the paper together with the dataset DOI/version.

In [5]:
import os, sys, json, warnings, importlib
PROJECT_ROOT = os.path.abspath("..")
if not os.path.isdir(os.path.join(PROJECT_ROOT, "notebooks")):
    PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)
# Prefer a configured data root only when it actually contains the paired folders.
def _has_paired_data(root):
    if not root:
        return False
    return ((os.path.isdir(os.path.join(root, "train", "train", "GT")) and
             os.path.isdir(os.path.join(root, "train", "train", "NoisyLR"))) or
            (os.path.isdir(os.path.join(root, "GT")) and
             os.path.isdir(os.path.join(root, "NoisyLR"))))
if not _has_paired_data(os.environ.get("SEMICON_DATA")):
    os.environ["SEMICON_DATA"] = PROJECT_ROOT
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import config as C
C = importlib.reload(C)
# Print explicit results and data roots below
from utils.data import load_npy
from utils.experiment import load_per_image, LADDER
from utils.analysis import robustness_grid, finished_seeds, load_run_model, restore
from utils.degrade import degrade, fit_degradation
from utils import stats as S, figures as FG, metrics as MT
from utils import carinthia as CA
RUNS = os.path.join(PROJECT_ROOT, "results_kaggle")
DATA = PROJECT_ROOT
C.OUT = RUNS
C.RUNS = os.path.join(RUNS, "runs")
C.TABLES = os.path.join(RUNS, "tables")
C.FIGS = os.path.join(RUNS, "figures")
C.DATA_ROOT = DATA
print("RUNS =", RUNS)
print("DATA =", DATA)
if not os.path.isfile(os.path.join(RUNS, "split_manifest.json")):
    raise FileNotFoundError(f"Kaggle results split manifest not found under RUNS={RUNS}")
splits, man = C.get_splits()
fp = os.path.join(C.TABLES, "degradation_fit.json")
fit = json.load(open(fp)) if os.path.exists(fp) else fit_degradation(splits["train"])
LOOKS, SIGMA = fit["looks"], fit["sigma"]; print(f"degradation used everywhere: looks={LOOKS:.2f}, sigma={SIGMA:.4f} (estimated from TRAIN pairs)")
names = [c for c in ["baseline", "global_gc", "film_control", "full", "NAFNet-small"] if finished_seeds(C.RUNS, c)]
seeds_by = {c: finished_seeds(C.RUNS, c) for c in names}; print(seeds_by)
dev = C.device()

RUNS = d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow\results_kaggle
DATA = d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow
degradation used everywhere: looks=9.26, sigma=0.0425 (estimated from TRAIN pairs)
{'baseline': [0, 1, 2, 3], 'global_gc': [0, 1], 'film_control': [0, 1, 2, 3], 'full': [0, 1, 2, 3], 'NAFNet-small': [0, 1, 2, 3]}


## 1. Robustness on the held-out TEST split

In [6]:
clean = np.stack([load_npy(g) for g, _ in splits["test"]]).astype(np.float32)
print("test images:", len(clean), "(none of them seen in training/validation)")
grid = robustness_grid(C.RUNS, names, seeds_by, clean, LOOKS, SIGMA, noise_seeds=(0, 1) if C.MODE == "smoke" else (0, 1, 2), device=dev)
grid.to_csv(os.path.join(C.TABLES, "robustness_long.csv"), index=False)
agg = grid.groupby(["config", "condition", "img"])[["psnr", "ssim"]].mean().reset_index()         # average over seeds and noise draws
tab = grid.groupby(["config", "condition", "seed"]).psnr.mean().reset_index().groupby(["config", "condition"]).psnr.agg(["mean", "std"]).round(3)
print(tab.unstack("condition").to_string())


test images: 320 (none of them seen in training/validation)


KeyboardInterrupt: 

In [ ]:
rows = []
for cond in agg.condition.unique():
    P = {c: agg[(agg.config == c) & (agg.condition == cond)].sort_values("img").psnr.values for c in names}
    for ref in [r for r in ["baseline", "film_control"] if r in names]:
        t = S.compare_to_reference(P, ref, names); t["condition"] = cond; rows.append(t)
rt = pd.concat(rows, ignore_index=True); rt["p_wilcoxon_holm_global"] = S.holm(rt.p_wilcoxon.values)
print(rt[["condition", "config", "vs", "mean_diff", "ci95_lo", "ci95_hi", "p_wilcoxon_holm_global", "cohens_dz"]].round(4).to_string(index=False))
rt.to_csv(os.path.join(C.TABLES, "table_robustness_paired_tests.csv"), index=False)
sp = rt[(rt.condition == "speckle only") & (rt.config == "full")]
for _, r in sp.iterrows():
    ok = r.mean_diff > 0 and r.ci95_lo > 0 and r.p_wilcoxon_holm_global < 0.05
    print(f"'speckle-only advantage' of full vs {r.vs}: {'SUPPORTED' if ok else 'NOT SUPPORTED'} (diff {r.mean_diff:+.4f} dB, CI [{r.ci95_lo:+.4f},{r.ci95_hi:+.4f}])")

      condition       config           vs  mean_diff  ci95_lo  ci95_hi  p_wilcoxon_holm_global  cohens_dz
       combined    global_gc     baseline    -0.0393  -0.0444  -0.0340                  0.0000    -0.8098
       combined film_control     baseline     0.0215   0.0097   0.0342                  0.0653     0.1924
       combined         full     baseline     0.0904   0.0783   0.1028                  0.0000     0.7989
       combined     baseline film_control    -0.0215  -0.0342  -0.0097                  0.0653    -0.1924
       combined    global_gc film_control    -0.0608  -0.0741  -0.0486                  0.0000    -0.5220
       combined         full film_control     0.0689   0.0552   0.0823                  0.0000     0.5560
downsample only    global_gc     baseline    -0.1635  -0.1934  -0.1357                  0.0000    -0.6237
downsample only film_control     baseline     0.0557   0.0311   0.0814                  0.0067     0.2424
downsample only         full     baseline    -

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6)); conds = list(agg.condition.unique()); w = 0.8 / len(names)
for k, c in enumerate(names):
    m, lo, hi = [], [], []
    for cond in conds:
        v = agg[(agg.config == c) & (agg.condition == cond)].psnr.values; m.append(v.mean()); l, h = S.bootstrap_ci(v, 2000); lo.append(v.mean() - l); hi.append(h - v.mean())
    ax.bar(np.arange(len(conds)) + k * w, m, w, yerr=[lo, hi], capsize=2, label=c)
ax.set_xticks(np.arange(len(conds)) + 0.4 - w / 2); ax.set_xticklabels(conds); ax.set_ylabel("PSNR (dB)"); ax.legend(ncol=3, fontsize=8)
ax.set_ylim(min(agg.psnr.mean() - 6, ax.get_ylim()[0]), None); ax.set_title("Per-degradation robustness - test split (95% bootstrap CI over images)")
FG.save(fig, C.FIGS, "fig_robustness", {"split": "test", "n_images": int(len(clean)), "looks": LOOKS, "sigma": SIGMA, "configs": names, "seeds": seeds_by})

## 2. Out-of-distribution SEM images (Carinthia-S)
Download the dataset (Zenodo record 16895427 - confirm version/DOI on the record page), unzip, and set `SEMICON_CARINTHIA`. Quote in the paper: DOI + version, number of images, selection procedure (seeded random sample), the manifest file, and the degradation parameters.

In [ ]:
root = C.CARINTHIA_ROOT
if C.SMOKE:
    root = os.path.join(C.OUT, "synthetic_carinthia"); CA.make_synthetic_carinthia(root, n=24, size=64)
OOD = os.path.isdir(root); SIZE = 64 if C.SMOKE else 256
if not OOD:
    print("Carinthia-S folder not found:", root, "\n-> download it and set SEMICON_CARINTHIA; the OOD section is skipped. DO NOT substitute other images.")
else:
    items = CA.load_pairs(root, n=100 if C.MODE == "full" else 30, seed=0, size=SIZE, manifest_path=os.path.join(C.TABLES, "carinthia_manifest.json"))
    print(len(items), "verified SEM images; first files:", [i["name"] for i in items[:3]], "| manifest ->", os.path.join(C.TABLES, "carinthia_manifest.json"))
    assert not any(CA.looks_like_mask(np.round(i["image"] * 255)) for i in items), "a label map slipped through"
    clean_o = np.stack([i["image"] for i in items]).astype(np.float32)
    deg_o = np.stack([degrade(clean_o[k], LOOKS, SIGMA, seed=k) for k in range(len(clean_o))])
    xt, gt = torch.from_numpy(deg_o)[:, None].to(dev), torch.from_numpy(clean_o)[:, None].to(dev)
    rows = []
    for c in names:
        for sd in seeds_by[c]:
            model, cfg, _ = load_run_model(C.RUNS, c, sd, dev)
            p = restore(model, cfg, xt).clamp(0, 1)
            for k in range(len(items)):
                rows.append(dict(config=c, seed=sd, img=k, name=items[k]["name"], psnr=float(MT.psnr(p[k:k+1], gt[k:k+1])), ssim=float(MT.ssim(p[k:k+1], gt[k:k+1]))))
    od = pd.DataFrame(rows); od.to_csv(os.path.join(C.TABLES, "ood_per_image.csv"), index=False)
    s = od.groupby(["config", "img"])[["psnr", "ssim"]].mean().reset_index()
    out = []
    for c in names:
        v = s[s.config == c]; lo, hi = S.bootstrap_ci(v.psnr.values, 2000)
        out.append(dict(config=c, n_images=len(v), PSNR=round(v.psnr.mean(), 3), PSNR_ci=(round(lo, 3), round(hi, 3)), SSIM=round(v.ssim.mean(), 4), SSIM_std_over_images=round(v.ssim.std(), 4)))
    print(pd.DataFrame(out).to_string(index=False)); pd.DataFrame(out).to_csv(os.path.join(C.TABLES, "table_ood.csv"), index=False)
    P = {c: s[s.config == c].sort_values("img").psnr.values for c in names}
    t = S.compare_to_reference(P, "baseline", names); print(t[["config", "mean_diff", "ci95_lo", "ci95_hi", "p_wilcoxon_holm", "significant_0.05"]].round(4).to_string(index=False)); t.to_csv(os.path.join(C.TABLES, "table_ood_paired.csv"), index=False)

Carinthia-S folder not found: d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow\data\Carinthia-S 
-> download it and set SEMICON_CARINTHIA; the OOD section is skipped. DO NOT substitute other images.


In [ ]:
if OOD:
    k0 = int(np.argsort(s[s.config == names[0]].psnr.values)[len(items) // 2])           # median image of the first model: fixed rule, not cherry-picked
    sd0 = seeds_by["full"][0] if "full" in names else seeds_by[names[0]][0]
    arrs = [("Clean (SEM)", clean_o[k0]), ("Degraded (bilinear x2)", np.clip(np.kron(deg_o[k0], np.ones((2, 2))) if deg_o.shape[-1] * 2 == clean_o.shape[-1] else deg_o[k0], 0, 1))]
    for c in names:
        model, cfg, _ = load_run_model(C.RUNS, c, seeds_by[c][0], dev)
        arrs.append((c, restore(model, cfg, xt[k0:k0+1]).clamp(0, 1)[0, 0].cpu().numpy()))
    FG.qualitative_grid(arrs, C.FIGS, "fig_ood_qualitative", {"dataset": "Carinthia-S", "file": items[k0]["name"], "sha": items[k0]["sha"], "selection": "median-PSNR image", "seed": seeds_by[names[0]][0], "looks": LOOKS, "sigma": SIGMA})
    print("saved fig_ood_qualitative for", items[k0]["name"])